# Cost Management

## AWS Free Tier

The **AWS Free Tier** provides free access to many services for 12 months after account creation. It includes:

- EC2: 750 hours/month of t2.micro instances
- S3: 5 GB of storage
- RDS: 750 hours/month of db.t2.micro instances
- Lambda: 1 million requests/month
- DynamoDB: 25 GB of storage
- CloudWatch: 10 custom metrics

After 12 months or usage limits, you pay standard rates. Always monitor usage to avoid unexpected charges.

## Pricing Models

**On-Demand** pricing charges per hour/second of usage. No upfront commitment. Best for variable workloads.

**Reserved Instances (RIs)** offer discounts (up to 72%) for 1 or 3-year commitments. Best for predictable, steady-state workloads.

**Spot Instances** offer up to 90% discounts for spare capacity. Can be interrupted with 2-minute notice. Best for fault-tolerant, flexible workloads.

**Savings Plans** provide discounts on compute usage across EC2, Lambda, and Fargate. More flexible than RIs.

## Cost Explorer

**Cost Explorer** visualizes your AWS spending. You can filter by service, region, or tag. It shows trends and forecasts future costs.

Use Cost Explorer to identify expensive services and optimize spending.

## Budgets

**AWS Budgets** lets you set spending limits and receive alerts when approaching or exceeding them. You can set budgets by service, region, or tag.

## Hands-On: Monitor Costs

View billing dashboard:

```bash
aws ce get-cost-and-usage --time-period Start=2024-01-01,End=2024-01-31 \
  --granularity MONTHLY --metrics "UnblendedCost" --group-by Type=DIMENSION,Key=SERVICE
```

Create a budget:

```bash
aws budgets create-budget --account-id ACCOUNT_ID \
  --budget '{
    "BudgetName": "monthly-budget",
    "BudgetLimit": {"Amount": "100", "Unit": "USD"},
    "TimeUnit": "MONTHLY",
    "BudgetType": "COST"
  }'
```

Create budget notification:

```bash
aws budgets create-notification --account-id ACCOUNT_ID \
  --budget-name monthly-budget \
  --notification '{
    "NotificationType": "ACTUAL",
    "ComparisonOperator": "GREATER_THAN",
    "Threshold": 80,
    "ThresholdType": "PERCENTAGE"
  }' \
  --subscribers '[{"SubscriptionType": "EMAIL", "Address": "user@example.com"}]'
```

Get cost and usage:

```bash
aws ce get-cost-and-usage --time-period Start=2024-01-01,End=2024-01-31 \
  --granularity DAILY --metrics "UnblendedCost"
```

## Python Boto3 Example

In [ ]:
import boto3
from datetime import datetime, timedelta

ce = boto3.client('ce')

# Get cost and usage
end_date = datetime.now().date()
start_date = end_date - timedelta(days=30)

response = ce.get_cost_and_usage(
    TimePeriod={
        'Start': start_date.isoformat(),
        'End': end_date.isoformat()
    },
    Granularity='DAILY',
    Metrics=['UnblendedCost'],
    GroupBy=[
        {'Type': 'DIMENSION', 'Key': 'SERVICE'}
    ]
)

for result in response['ResultsByTime']:
    print(f"Date: {result['TimePeriod']['Start']}")
    for group in result['Groups']:
        service = group['Keys'][0]
        cost = group['Metrics']['UnblendedCost']['Amount']
        print(f"  {service}: ${cost}")

## Reserved Instances

Purchase RIs for predictable workloads:

```bash
aws ec2 describe-reserved-instances-offerings \
  --instance-type t3.micro --filters Name=scope,Values=us-east-1
```

## Spot Instances

Launch Spot instances for cost savings:

```bash
aws ec2 request-spot-instances --spot-price 0.05 \
  --instance-count 1 --type one-time \
  --launch-specification '{
    "ImageId": "ami-0c55b159cbfafe1f0",
    "InstanceType": "t3.micro",
    "KeyName": "my-key"
  }'
```

## Terraform Example

```hcl
resource "aws_instance" "on_demand" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"

  tags = {
    Name = "on-demand"
  }
}

resource "aws_instance" "spot" {
  ami                    = "ami-0c55b159cbfafe1f0"
  instance_type          = "t3.micro"
  spot_price             = "0.05"
  instance_interruption_behavior = "terminate"

  tags = {
    Name = "spot"
  }
}

resource "aws_ec2_reserved_instance" "example" {
  instance_type           = "t3.micro"
  offering_class          = "standard"
  offering_type           = "ALL_UPFRONT"
  purchase_term           = "ONE_YEAR"
  availability_zone       = "us-east-1a"
  instance_count          = 1
}
```

## Cost Optimization Tips

1. Use the free tier for learning and development
2. Right-size instances (don't over-provision)
3. Use Reserved Instances for predictable workloads
4. Use Spot Instances for fault-tolerant workloads
5. Enable S3 lifecycle policies to move old data to cheaper storage
6. Delete unused resources (EBS volumes, snapshots, etc.)
7. Use CloudWatch to monitor resource usage
8. Set up budgets and alerts

## Quiz 1

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the AWS Free Tier?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="0">
      <span>A permanent free service tier</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="1">
      <span>Free access to services for 12 months after account creation</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="2">
      <span>A discount on all services</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="3">
      <span>A trial period for new users</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What are Reserved Instances?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="0">
      <span>Discounted instances for 1 or 3-year commitments</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="1">
      <span>Instances that can be interrupted</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="2">
      <span>Instances for development only</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="3">
      <span>Instances with guaranteed uptime</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What are Spot Instances?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="0">
      <span>Instances for long-term commitments</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="1">
      <span>Instances for production workloads</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="2">
      <span>Discounted instances for spare capacity that can be interrupted</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="3">
      <span>Instances with guaranteed uptime</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is Cost Explorer?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="0">
      <span>A service to purchase instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="1">
      <span>A tool to visualize and analyze AWS spending</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="2">
      <span>A service to manage budgets</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="3">
      <span>A service to optimize instance types</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What are AWS Budgets?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="0">
      <span>Tools to set spending limits and receive alerts</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="1">
      <span>Discounts on services</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="2">
      <span>Reserved instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="3">
      <span>Cost analysis reports</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>